# Import and Shared functions

In [48]:
import os, sys
import ast
import math
import re
import pandas as pd
import numpy as np
import ephem
from scipy import signal
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from dateutil import parser as dateutil_parser
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib
import kinematics
import control
import analyse_helpers
importlib.reload(kinematics)
importlib.reload(control)
importlib.reload(analyse_helpers)
from kinematics import calc_parallactic_angle, azaltroll_to_theta, apply_mechanical_corrections, azaltroll_to_q, MountModelParams
from control import theta_to_jacobian, PecMode
from quaternion import Q as Quaternion
from analyse_helpers import resolve_log_files

def r2_score(y, yhat):
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    return 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

def r2_quality(r2):
    return (
        "excellent" if r2>0.95 else 
        "good" if r2>0.8 else 
        "moderate" if r2>0.5 else 
        "weak" if r2>0.25 else 
        "poor")

def pec_quality_db(snr_db):
    return (
        "excellent" if snr_db > 30 else
        "good"      if snr_db > 20 else
        "moderate"  if snr_db > 10 else
        "weak"      if snr_db > 5  else
        "poor"
    )

def parse_val(v):
    v = v.strip()
    try:
        return float(v)
    except ValueError:
        return v
    
def parse_pecconfig(line):
    """Extract the PECCONFIG line emitted once per session, if present."""
    m = re.search(
        r'PECCONFIG mode,(\w+),n_harmonics,(\d+),T,([\d.]+),tau_sec,([\d.]+),min_dt_sec,([\d.]+)',
        line
    )
    if not m:
        return None
    mode, H, T, tau, min_dt = m.groups()
    return dict(mode=mode, H=int(H), T=float(T), tau=float(tau), min_dt=float(min_dt))

def parse_pec_line(line):
    """
    Split 'TIMESTAMP INFO PECLOG {dict}' and literal_eval the trailing dict, flattening any
    list-valued field (currently just ra_model/dec_model -- index 0 is the DC/steady-state
    component, 1..n_harmonics are that harmonic's contribution) into '<key>_<i+1>' columns,
    same convention as analyse_kf_pid.ipynb's KFLOG/PIDLOG parser. Matches control.py's
    _pec_log(), which logs one such dict per guide-sync cycle.
    """
    if " PECLOG " not in line:
        return None
    ts, _, body = line.partition(" PECLOG ")
    ts = ts.split(" INFO")[0].strip()
    try:
        payload = ast.literal_eval(body.strip())
    except (ValueError, SyntaxError):
        return None
    rec = {"timestamp": ts}
    for key, val in payload.items():
        if isinstance(val, list):
            for i, v in enumerate(val):
                rec[f"{key}_{i+1}"] = v
        else:
            rec[key] = val
    return rec

def load_pec(log_filenames, log_dir='.'):
    """
    Parse PECCONFIG/PECLOG lines from one or more rotated driver logs into a single
    DataFrame, plus the session's pec_config dict. Accepts a single filename/path, a glob
    pattern (e.g. 'alpaca.log*'), or an explicit list of filenames/paths/globs -- all rows
    are concatenated and re-sorted by timestamp, so file order/naming doesn't matter. The
    first PECCONFIG line found (across all files, in resolution order) wins, since it's
    emitted once per session and shouldn't change across a session's rotated logs. See
    analyse_helpers.resolve_log_files() for how log_filenames is resolved against log_dir.
    """
    paths = resolve_log_files(log_filenames, log_dir=log_dir)
    if not paths:
        raise FileNotFoundError(f"No log files matched: {log_filenames!r} (log_dir={log_dir!r})")

    rows = []
    pec_config = None
    for log_path in paths:
        if not os.path.exists(log_path):
            raise FileNotFoundError(f"log_path does not exist: {log_path!r}")
        if os.path.getsize(log_path) == 0:
            raise ValueError(f"log_path exists but is empty (0 bytes): {log_path!r}")
        # encoding='utf-8': the driver's RotatingFileHandler writes utf-8 explicitly (log.py), but
        # open() without an explicit encoding falls back to the OS default -- cp1252 on Windows --
        # which can silently corrupt any non-ASCII content instead of raising.
        with open(log_path, encoding="utf-8") as f:
            for line in f:
                if 'PECCONFIG' in line:
                    if pec_config is None:
                        cfg = parse_pecconfig(line)
                        if cfg is not None:
                            pec_config = cfg
                    continue
                if ' PECLOG ' not in line:
                    continue
                rec = parse_pec_line(line)
                if rec is None:
                    continue
                rows.append(rec)

    if not rows:
        raise ValueError(f"No PECLOG lines found in {paths!r}")

    df = pd.DataFrame(rows)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    # Dedup on the FULL row, not just timestamp -- guards against overlapping rotated log
    # segments producing the same record twice, same convention as analyse_kf_pid.ipynb.
    df = df.sort_values('timestamp', kind='stable').drop_duplicates().reset_index(drop=True)
    df['t_sec'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
    return df, pec_config

# Load data

In [49]:
# ── Choose correct log path (last set log_filenames is what is used) ──────────────────────────────────
# Single file:            'alpaca.log'                     (relative to LOG_DIR below)
# All rotated files:      'alpaca.log*'                    (glob -- handles a multi-hour run spanning several files)
# Explicit file list:     ['alpaca.log.2', 'alpaca.log.1', 'alpaca.log']
# A full/relative path (as before) also still works and bypasses LOG_DIR: '../logs/alpaca.log'
LOG_DIR = '../logs'   # base directory log_filenames below are resolved against

log_filenames = 'alpaca.sga_omega_cent.log' # Omega Centuri 1st session
log_filenames = 'alpaca.sga_lagoon.log'     # Lagoon Nebula 2nd session
log_filenames = 'alpaca.sga_pulsepec.log'   # Pulse Guiding with PEC on Beehive cluster
log_filenames = 'alpaca.sga_eagle.log'      # Sync Guiding with PEC on Eagle Nebula
log_filenames = 'alpaca.sga_catspaw.log'      # Sync Guiding with PEC on Eagle Nebula
log_filenames = 'alpaca.pecSlide.acrux2.log'
log_filenames = 'vlogs/alpaca.log.4'       
log_filenames = 'alpaca.h1.h0.log'
log_filenames = 'alpaca.pec_rls_h2.log'
#log_filenames = 'alpaca.pec_ema.log'
#log_filenames = 'alpaca.pec_rls_7hrs.log'
#log_filenames = 'alpaca.pec_rls_6hrs.log'
#log_filenames = 'alpaca.mark_Beta3_08_02.log'
log_filenames = 'alpaca.pec_master.log'
log_filenames = 'alpaca.mark_Beta3_08_02.log'
#log_filenames = 'alpaca.pecSin.chicken.log' # PHD2 guiding, somewhat ok
log_filenames = 'alpaca.log'
log_filenames = ['alpaca.soak_pec_Beta4.3_08_30a*.log']  # glob inside a list, like analyse_kf_pid.ipynb
log_filenames = ['alpaca.soak_pec_Beta4.3_08_29m*.log']  # glob inside a list, like analyse_kf_pid.ipynb
LOG_DIR = '../logs/logs'


In [50]:
# ── Read and Parse PECLOG data from log file(s) ───────────────────────────────────────────────────────────
resolved_files = resolve_log_files(log_filenames, log_dir=LOG_DIR)
print(f"Loading {len(resolved_files)} file(s) from LOG_DIR={LOG_DIR!r}:")
for p in resolved_files:
    print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
print()

df, pec_config = load_pec(log_filenames, log_dir=LOG_DIR)

# Fallback if this log predates PECCONFIG (older sessions) — infer what we can
if pec_config is None:
    ra_model_cols = [c for c in df.columns if re.fullmatch(r'ra_model_\d+', c)]
    n_model_terms = len(ra_model_cols)
    if n_model_terms == 0:
        print("WARNING: no PECCONFIG line and no ra_model/dec_model columns found — "
              "cannot determine mode reliably. Defaulting to RLS/H=0, verify manually.")
        pec_config = dict(mode='rls', H=0, T=34*60, tau=21*60, min_dt=0.05)
    else:
        pec_config = dict(mode='rls', H=max(n_model_terms - 1, 0), T=34*60, tau=21*60, min_dt=0.05)
    print("No PECCONFIG line found — inferred:", pec_config)
else:
    print("PEC config from log:", pec_config)

print(f"Duration: {df['t_sec'].iloc[-1]/3600:.2f} hours")
print(f"N samples: {len(df)}")
print(f"Az range: {df['az'].min():.3f} to {df['az'].max():.3f}")
print()
df.columns


Loading 1 file(s) from LOG_DIR='../logs/logs':
  ../logs/logs/alpaca.soak_pec_Beta4.3_08_29m.log  (36.3 MB)

No PECCONFIG line found — inferred: {'mode': 'rls', 'H': 2, 'T': 2040, 'tau': 1260, 'min_dt': 0.05}
Duration: 0.84 hours
N samples: 29
Az range: 334.117 to 346.839



Index(['timestamp', 'n', 'inhibit_1', 'inhibit_2', 'r2_1', 'r2_2', 'rmse_1',
       'rmse_2', 'resid_1', 'resid_2', 'fit_rate_1', 'fit_rate_2',
       'applied_rate_1', 'applied_rate_2', 'pec_accum_1', 'pec_accum_2',
       'total_accum_1', 'total_accum_2', 'ra_model_1', 'ra_model_2',
       'ra_model_3', 'dec_model_1', 'dec_model_2', 'dec_model_3', 'az', 'alt',
       'roll', 'lambda_1', 'lambda_2', 'pec_active', 'age_518', 't_sec'],
      dtype='str')

# Optional Save to CSV

In [51]:
# df.to_csv('data.csv', index=False)

# Mount Orientation (Az, Alt, Roll) vs Time

In [52]:
# ── plot Az, Alt, Roll ───────────────────────────────────────────────────────────
xdata = df['t_sec'] / 60
ydata = [
    #row, dataset,            name,                color
    (1,   df['az'],    'Azimuth (°)',  'royalblue'),
    (2,   df['alt'],   'Altitude (°)', 'orange'),
    (3,   df['roll'],  'Roll (°)',     'mediumseagreen'),
]
rows = len(set([row for row, y,name,color in ydata]))
fig = make_subplots(rows=rows, cols=1, shared_xaxes=True,
    subplot_titles=[name for row, y,name,color in ydata],
    vertical_spacing=0.12)

for (row, y, name, color) in ydata:
    fig.add_trace(go.Scatter(x=xdata, y=y,
        mode='lines+markers', line=dict(color=color, width=0.8),
        name=name), row=row, col=1)
    fig.update_yaxes(title_text=name, row=row, col=1)

fig.update_xaxes(title_text='Time (minutes)', row=3, col=1)
fig.update_layout(height=800, width=1200, template="plotly_dark",
        title='Range of Mount Orientation over session')
fig.show()

# PEC Outcome Overview

The real test of whether PEC is working: is `resid` (this sync's raw guide error, in arcmin --
`resid_1`=RA, `resid_2`=Dec after the standard list-flattening) trending toward zero and getting
less noisy over the session? This is the ground-truth guiding outcome PEC is trying to improve,
distinct from `total_accum` (which accumulates continuously by design and never shrinks -- see the
section below) and from `rmse`/`r2` (which describe the *model's own fit quality*, not the
resulting guiding error). Per control.py's `_pec_log()`: "should shrink as PEC improves."

Note `n` (and so `resid`) resets whenever the PEC model resets (goto/rotate/jog/stop-tracking/config
change -- see the reset-aware analysis further below), so a session spanning multiple resets will
show the residual jump back up at each reset before resuming its downward trend within that segment.

In [53]:
# ── PEC Outcome Overview: are sync-guide residuals shrinking over the session? ─────────────
resid = df.dropna(subset=['resid_1', 'resid_2']).copy()
resid['resid_ra_arcsec']  = resid['resid_1'] * 60
resid['resid_dec_arcsec'] = resid['resid_2'] * 60
resid = resid.sort_values('t_sec').reset_index(drop=True)

ROLL_N = max(5, len(resid) // 20)   # ~20 points of smoothing across the session, floor of 5 cycles
resid['ra_rms_roll']  = resid['resid_ra_arcsec'].rolling(ROLL_N, min_periods=1).apply(lambda x: np.sqrt(np.mean(x**2)))
resid['dec_rms_roll'] = resid['resid_dec_arcsec'].rolling(ROLL_N, min_periods=1).apply(lambda x: np.sqrt(np.mean(x**2)))

def trend_summary(t, y, label):
    """OLS slope of y (arcsec) vs t (sec) -- negative means shrinking/improving -- plus a
    first-fifth vs last-fifth RMS comparison, which is more robust to a single noisy cycle
    than the endpoints alone."""
    if len(t) < 4:
        print(f"{label}: not enough sync cycles ({len(t)}) for a trend")
        return
    slope, intercept = np.polyfit(t, y, 1)
    yhat = np.polyval([slope, intercept], t)
    r2 = r2_score(y, yhat)
    n = len(y)
    k = max(1, n // 5)
    rms_first = np.sqrt(np.mean(y[:k]**2))
    rms_last  = np.sqrt(np.mean(y[-k:]**2))
    pct_change = 100 * (rms_last - rms_first) / rms_first if rms_first != 0 else np.nan
    direction = "decreasing -- PEC improving" if slope < 0 else "increasing -- PEC not converging / worsening"
    print(f"=== {label} sync residual outcome (N={n} cycles) ===")
    print(f"Linear trend:         {slope*3600:+8.3f} arcsec/hr   R²={r2:.3f} ({r2_quality(r2)} fit) -- {direction}")
    print(f"First {k} cycles RMS:   {rms_first:8.3f} arcsec")
    print(f"Last  {k} cycles RMS:   {rms_last:8.3f} arcsec")
    print(f"Change:                {pct_change:+7.1f}%  ({'improvement' if pct_change < 0 else 'regression'})")
    print()

trend_summary(resid.t_sec.values, resid.resid_ra_arcsec.values,  "RA")
trend_summary(resid.t_sec.values, resid.resid_dec_arcsec.values, "Dec")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=["RA sync-guide residual (arcsec) -- should trend toward 0 as PEC learns",
                     "Dec sync-guide residual (arcsec) -- should trend toward 0 as PEC learns"],
    vertical_spacing=0.1)

fig.add_trace(go.Scatter(x=resid.t_sec/60, y=resid.resid_ra_arcsec, mode='markers', name='RA resid (raw)',
    marker=dict(size=4, color='royalblue', opacity=0.4)), row=1, col=1)
fig.add_trace(go.Scatter(x=resid.t_sec/60, y=resid.ra_rms_roll, mode='lines', name=f'RA rolling RMS (n={ROLL_N})',
    line=dict(color='orange', width=2)), row=1, col=1)

fig.add_trace(go.Scatter(x=resid.t_sec/60, y=resid.resid_dec_arcsec, mode='markers', name='Dec resid (raw)',
    marker=dict(size=4, color='mediumseagreen', opacity=0.4)), row=2, col=1)
fig.add_trace(go.Scatter(x=resid.t_sec/60, y=resid.dec_rms_roll, mode='lines', name=f'Dec rolling RMS (n={ROLL_N})',
    line=dict(color='orange', width=2)), row=2, col=1)

fig.add_hline(y=0, line=dict(color='gray', width=1, dash='dot'), row=1, col=1)
fig.add_hline(y=0, line=dict(color='gray', width=1, dash='dot'), row=2, col=1)

fig.update_xaxes(title_text='Time (minutes)', row=2, col=1)
fig.update_yaxes(title_text='arcsec', row=1, col=1)
fig.update_yaxes(title_text='arcsec', row=2, col=1)
fig.update_layout(height=700, width=1200, template='plotly_dark',
    title='PEC Outcome: Sync-Guide Residual Trend (raw resid + rolling RMS)',
    hovermode='x unified', legend=dict(groupclick='toggleitem'))
fig.show()

=== RA sync residual outcome (N=29 cycles) ===
Linear trend:         +105.473 arcsec/hr   R²=0.164 (poor fit) -- increasing -- PEC not converging / worsening
First 5 cycles RMS:     25.736 arcsec
Last  5 cycles RMS:     77.350 arcsec
Change:                 +200.6%  (regression)

=== Dec sync residual outcome (N=29 cycles) ===
Linear trend:           -7.321 arcsec/hr   R²=0.002 (poor fit) -- decreasing -- PEC improving
First 5 cycles RMS:     32.747 arcsec
Last  5 cycles RMS:     41.170 arcsec
Change:                  +25.7%  (regression)



# Analysis of PECLOG

In [54]:
# ── plot rows 1–4 ───────────────────────────────────────────────────────────
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from control import PecAxis, PecInhibit, PecMode

# ── calc INTERVAL (diagnostic only — no longer drives lambda/alpha directly) ──
ra_times = df.dropna(subset=['resid_1'])['t_sec']
INTERVAL = ra_times.diff().dropna().median()

# ── instantiate from parsed session config ────────────────────────────────
T        = pec_config['T']
H        = pec_config['H']
TAU      = pec_config['tau']
MIN_DT   = pec_config['min_dt']
MODE     = PecMode(pec_config['mode']) if pec_config['mode'] in ('rls', 'ema') else PecMode.RLS
VAR      = 0.05
SSE      = 0.15
COLORS   = ['green', 'purple', 'orange', 'red', 'cyan', 'magenta']
DEG_S_TO_ARCMIN_HR = 3600 * 60   # deg/sec to arcmin/hr

ra  = PecAxis(T=T, n_harmonics=H, mode=MODE, tau=TAU, min_dt=MIN_DT)
dec = PecAxis(T=T, n_harmonics=H, mode=MODE, tau=TAU, min_dt=MIN_DT)
var_alpha = VAR
sse_alpha = SSE

ra.reset_seed(df.iloc[0].total_accum_1 / 60)
dec.reset_seed(df.iloc[0].total_accum_2 / 60)

t0 = 0.0            # local time origin — shifts forward on every detected PEC reset
prev_n = df.iloc[0].n

show_harmonics = (MODE == PecMode.RLS and H > 0)

records = []
reset_events = []   # for plotting/inspection — t_min of each detected reset
reset_indices = []  # for fixing gradient

for i, row in df.iterrows():
    t_abs     = row.t_sec
    ra_cumul  = row.total_accum_1  / 60
    dec_cumul = row.total_accum_2 / 60

    # ── detect a PEC reset from the guide counter rolling back ──
    if row.n < prev_n:
        ra.reset()
        dec.reset()
        ra.reset_seed(ra_cumul)
        dec.reset_seed(dec_cumul)
        t0 = t_abs                      # restart local clock — matches driver's _pec_t0 reset
        reset_events.append(t_abs / 60)
        reset_indices.append(i)
    prev_n = row.n

    t = t_abs - t0                      # local time, relative to most recent reset

    ra.ingest_accum(ra_cumul, t, var_alpha, sse_alpha)
    dec.ingest_accum(dec_cumul, t, var_alpha, sse_alpha)

    records.append(dict(
        t_min    = t_abs / 60,          # keep x-axis on session-absolute time for plotting continuity
        n        = row.n,
        ra_rate  = ra.predicted_rate(t)  * DEG_S_TO_ARCMIN_HR,
        dec_rate = dec.predicted_rate(t) * DEG_S_TO_ARCMIN_HR,
        ra_drift = ra.dc_rate()   * DEG_S_TO_ARCMIN_HR,
        dec_drift= dec.dc_rate()  * DEG_S_TO_ARCMIN_HR,
        **{f'ra_h{h}':  ra.harmonic_rate(h)  * DEG_S_TO_ARCMIN_HR for h in range(1, H+1)},
        **{f'dec_h{h}': dec.harmonic_rate(h) * DEG_S_TO_ARCMIN_HR for h in range(1, H+1)},
        ra_r2    = ra.r2,
        dec_r2   = dec.r2,
        ra_rmse  = ra.rmse_arcmin(),
        dec_rmse = dec.rmse_arcmin(),
        ra_pred  = ra.predicted_accum(t)  * 60,
        dec_pred = dec.predicted_accum(t) * 60,
        ra_cumul = ra_cumul  * 60,
        dec_cumul= dec_cumul * 60,
        ra_resid_fit  = (ra_cumul*60)  - ra.predicted_accum(t)*60,
        dec_resid_fit = (dec_cumul*60) - dec.predicted_accum(t)*60,
        ra_lambda  = ra.lam,
        dec_lambda = dec.lam,
    ))

res = pd.DataFrame(records)
t   = res.t_min

# ── empirical (actual) drift rate ──
res['ra_actual_rate']  = np.full(len(res), np.nan)
res['dec_actual_rate'] = np.full(len(res), np.nan)

reset_indices = sorted(set([0] + reset_indices + [len(res)]))
for start, end in zip(reset_indices[:-1], reset_indices[1:]):
    if end - start < 2:
        continue   # need at least 2 points to take a gradient
    seg = res.iloc[start:end]
    seg_t = seg.t_min.values * 60           # ra_cumul in arcmin; seg_t in seconds; ra_actual_rate in arcmin/hr
    res.loc[seg.index, 'ra_actual_rate']  = np.gradient(seg.ra_cumul.values,  seg_t) * 3600
    res.loc[seg.index, 'dec_actual_rate'] = np.gradient(seg.dec_cumul.values, seg_t) * 3600

mode_label = MODE.value.upper()
title_text = (f'PEC Model (mode={mode_label}   Hₙ={H}   Period₁={T/60:.0f} min   '
              f'dt_med={INTERVAL:.1f}s   τ={TAU:.0f}s   sse_α={SSE}   var_α={VAR})')

# ── plot rows 1–4 ───────────────────────────────────────────────────────────
n_rows_top = 4
row_titles_top = [
    'PEC guide counter (resets to 1 after goto)',            'PEC guide counter (resets to 1 after goto)',
    'RA total_accum and fit (arcmin)',                       'Dec total_accum and fit (arcmin)',
    'RA predicted vs actual drift rate (arcmin/hr)',         'Dec predicted vs actual drift rate (arcmin/hr)',
    'RA predicted rate breakdown: DC + harmonics (arcmin/hr)','Dec predicted rate breakdown: DC + harmonics (arcmin/hr)',
]

fig_top = make_subplots(
    rows=n_rows_top, cols=2,
    shared_xaxes=True,
    subplot_titles=tuple(row_titles_top),
    vertical_spacing=0.08,
)

def add(fig, row, col, traces):
    for tr in traces:
        fig.add_trace(tr, row=row, col=col)

# row 1 — counter
add(fig_top, 1, 1, [go.Scatter(x=t, y=res.n, name='n', mode='lines+markers', line=dict(color='royalblue'))])
add(fig_top, 1, 2, [go.Scatter(x=t, y=res.n, name='n', mode='lines+markers', line=dict(color='royalblue'))])

# row 2 — cumul fit
add(fig_top, 2, 1, [
    go.Scatter(x=t, y=res.ra_cumul, name='RA actual',    mode='lines+markers', opacity=0.6, line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.ra_pred,  name='RA predicted', line=dict(color='red', width=2)),
])
add(fig_top, 2, 2, [
    go.Scatter(x=t, y=res.dec_cumul, name='Dec actual',    mode='lines+markers', opacity=0.6, line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.dec_pred,  name='Dec predicted', line=dict(color='red', width=2)),
])

# row 3 — predicted vs actual total drift rate
add(fig_top, 3, 1, [
    go.Scatter(x=t, y=res.ra_rate,        name='RA Predicted Rate', line=dict(color='orange')),
    go.Scatter(x=t, y=res.ra_actual_rate, name='RA Actual Rate',    mode='lines', opacity=0.6, line=dict(color='royalblue')),
])
add(fig_top, 3, 2, [
    go.Scatter(x=t, y=res.dec_rate,        name='Dec Predicted Rate', line=dict(color='orange')),
    go.Scatter(x=t, y=res.dec_actual_rate, name='Dec Actual Rate',    mode='lines', opacity=0.6, line=dict(color='royalblue')),
])

# row 4 — breakdown of predicted rate: DC + harmonics
add(fig_top, 4, 1, [
    go.Scatter(x=t, y=res.ra_drift, name='RA DC Component', line=dict(color='#636EFA', dash='dash')),
] + ([
    go.Scatter(x=t, y=res[f'ra_h{h}'], name=f'RA H{h} Component', line=dict(color=COLORS[(h-1) % len(COLORS)], dash='dash'))
    for h in range(1, H+1)
] if show_harmonics else []))
add(fig_top, 4, 2, [
    go.Scatter(x=t, y=res.dec_drift, name='Dec DC Component', line=dict(color='#636EFA', dash='dash')),
] + ([
    go.Scatter(x=t, y=res[f'dec_h{h}'], name=f'Dec H{h} Component', line=dict(color=COLORS[(h-1) % len(COLORS)], dash='dash'))
    for h in range(1, H+1)
] if show_harmonics else []))

fig_top.update_layout(
    height=300 * n_rows_top,
    title=dict(text=title_text, x=0.5),
    hovermode='x unified', template="plotly_dark",
    legend=dict(groupclick='toggleitem'),
)
fig_top.update_xaxes(title_text='Time (minutes)', row=n_rows_top)
fig_top.show()


In [55]:
# ── plot rows 5–6 ───────────────────────────────────────────────────────────
row5_min = 0
n_rows_bottom = 2
row_titles_bottom = [
    'RA fit R² quality',                      'Dec fit R² quality',
    'RA fit residuals and rmse (arcmin)',     'Dec fit residuals and rmse (arcmin)',
]

fig_bottom = make_subplots(
    rows=n_rows_bottom, cols=2,
    shared_xaxes=True,
    subplot_titles=tuple(row_titles_bottom),
    vertical_spacing=0.12,
)

def add(fig, row, col, traces):
    for tr in traces:
        fig.add_trace(tr, row=row, col=col)

# row 1 (of this fig) — R²
add(fig_bottom, 1, 1, [go.Scatter(x=t, y=res.ra_r2,  name='RA R²',  line=dict(color='royalblue'))])
add(fig_bottom, 1, 2, [go.Scatter(x=t, y=res.dec_r2, name='Dec R²', line=dict(color='royalblue'))])

# row 2 (of this fig) — residuals
add(fig_bottom, 2, 1, [
    go.Scatter(x=t, y=res.ra_resid_fit, name='RA Residuals', line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.ra_rmse,      name='RA rmse',      line=dict(color='red', dash='dash')),
])
add(fig_bottom, 2, 2, [
    go.Scatter(x=t, y=res.dec_resid_fit, name='Dec Residuals', line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.dec_rmse,      name='Dec rmse',      line=dict(color='red', dash='dash')),
])

fig_bottom.add_hrect(y0=row5_min, y1=0.5, fillcolor='rgba(255,0,0,0.15)', line_width=0, row=1, col=1)
fig_bottom.add_hrect(y0=row5_min, y1=0.5, fillcolor='rgba(255,0,0,0.15)', line_width=0, row=1, col=2)
fig_bottom.update_yaxes(range=[row5_min, 1.0], row=1, col=1)
fig_bottom.update_yaxes(range=[row5_min, 1.0], row=1, col=2)

fig_bottom.update_layout(
    height=300 * n_rows_bottom,
    title=dict(text=title_text, x=0.5),
    hovermode='x unified', template="plotly_dark",
    legend=dict(groupclick='toggleitem'),
)
fig_bottom.update_xaxes(title_text='Time (minutes)', row=n_rows_bottom)
fig_bottom.show()


# Cumulative Total Drift (total_accum) vs Time

In [56]:
# ── plot accumulated corrections single chart ───────────────────────────────────────────────────────────
fig = go.Figure()
fig.update_layout(
    height=600 , width=1800,
    title=dict(text="RA Axis - Cumulative Total Drift (total_accum) vs Time", x=0.5, font_size=32),
    xaxis=dict(title="Time (minutes)", title_font=dict(size=20)),
    yaxis=dict(title="Total Accumulated Drift (arcmin)", title_font=dict(size=20)),
    legend_font_size=20,
    hovermode='x unified', template="plotly_dark",
    legend=dict(groupclick='toggleitem'),
)
fig.add_trace(go.Scatter(x=t, y=res.ra_cumul, name='RA actual',    mode='lines+markers', opacity=0.6, line=dict(color='white')))
fig.add_trace(go.Scatter(x=t, y=res.ra_pred,  name='RA predicted', line=dict(color='green', width=2)))
fig.show()

fig = go.Figure()
fig.update_layout(
    height=600 , width=1800,
    title=dict(text="Dec Axis - Cumulative Total Drift (total_accum) vs Time", x=0.5, font_size=32),
    xaxis=dict(title="Time (minutes)", title_font=dict(size=20)),
    yaxis=dict(title="Total Accumulated Drift (arcmin)", title_font=dict(size=20)),
    legend_font_size=20,
    hovermode='x unified', template="plotly_dark",
    legend=dict(groupclick='toggleitem'),
)
fig.add_trace(go.Scatter(x=t, y=res.dec_cumul, name='Dec actual',    mode='lines+markers', opacity=0.6, line=dict(color='white')))
fig.add_trace(go.Scatter(x=t, y=res.dec_pred,  name='Dec predicted', line=dict(color='green', width=2)))
fig.show()

# Right Ascension Drift

In [57]:
t = df['t_sec'].values
dec = df['total_accum_1'].values
poly_coeffs, residuals, rank, sv, rcond = np.polyfit(t, dec, 1, full=True)
slope, intercept = poly_coeffs
dec_fit = np.polyval(poly_coeffs, t)
n = len(t)
rss = residuals[0] if len(residuals) > 0 else np.sum((dec - dec_fit)**2)
tss = np.sum((y - np.mean(dec))**2)
r2 = 1 - rss / tss if tss != 0 else float('nan')
rmse = np.sqrt(rss / n)
print(f"=== Right Ascension - Drift Fit Summary (N={n}) ===")
print(f"Slope (drift rate): {slope*3600:+8.3f} arcmin/hr")
print(f"Intercept:          {intercept:+8.3f} arcmin")
print(f"RMSE (noise):       {rmse:8.3f} arcmin")
print(f"R² (fit quality):   {r2:8.3f} {r2_quality(r2)} fit")

=== Right Ascension - Drift Fit Summary (N=29) ===
Slope (drift rate):  -21.028 arcmin/hr
Intercept:            +0.339 arcmin
RMSE (noise):          1.093 arcmin
R² (fit quality):      0.996 excellent fit


# Declination Drift

In [58]:
t = df['t_sec'].values
dec = df['total_accum_2'].values
poly_coeffs, residuals, rank, sv, rcond = np.polyfit(t, dec, 1, full=True)
slope, intercept = poly_coeffs
dec_fit = np.polyval(poly_coeffs, t)
n = len(t)
rss = residuals[0] if len(residuals) > 0 else np.sum((dec - dec_fit)**2)
tss = np.sum((y - np.mean(dec))**2)
r2 = 1 - rss / tss if tss != 0 else float('nan')
rmse = np.sqrt(rss / n)
print(f"=== Declination - Drift Fit Summary (N={n}) ===")
print(f"Slope (drift rate): {slope*3600:+8.3f} arcmin/hr")
print(f"Intercept:          {intercept:+8.3f} arcmin")
print(f"RMSE (noise):       {rmse:8.3f} arcmin")
print(f"R² (fit quality):   {r2:8.3f} {r2_quality(r2)} fit")


=== Declination - Drift Fit Summary (N=29) ===
Slope (drift rate):  +22.602 arcmin/hr
Intercept:            +8.708 arcmin
RMSE (noise):          0.636 arcmin
R² (fit quality):      1.000 excellent fit


# Right Ascension Period

In [59]:
from scipy.signal import periodogram
import numpy as np

# RA: the periodic PEC signal
ra = df['total_accum_1'].values

# Remove linear drift (same idea as DEC fit)
ra_detrended = ra - np.polyval(np.polyfit(t, ra, 1), t)

# Sampling frequency
fs = 1 / np.median(np.diff(t))

# Periodogram
freqs, power = periodogram(ra_detrended, fs=fs)

# Ignore zero frequency and periods > 100 min 
max_period_sec = 120 * 60
min_freq = 1 / max_period_sec
valid = (freqs >= min_freq)
freqs = freqs[valid]
power = power[valid]

# Peak detection
peak_idx = np.argmax(power)
peak_freq = freqs[peak_idx]
peak_power = power[peak_idx]

worm_period_sec = 1 / peak_freq

# Noise floor estimate (median is robust)
mask = np.ones_like(power, dtype=bool)
window = 3  # exclude ±3 bins around peak
mask[max(0, peak_idx-window):peak_idx+window+1] = False
noise_floor = np.median(power[mask])

# Signal-to-noise ratio
snr = peak_power / noise_floor if noise_floor > 0 else np.inf
snr_db = 10 * np.log10(peak_power / noise_floor)

# Estimate amplitude (rough, from detrended signal)
amp = (np.max(ra_detrended) - np.min(ra_detrended)) / 2

# Output
n = len(ra)

print(f"=== Right Ascension - PEC Analysis (N={n}) ===")
print(f"Worm period:        {worm_period_sec/60:8.3f} min")
print(f"Amplitude:          {amp*60:8.3f} arcmin")
print(f"Peak power:         {10*np.log10(peak_power):8.3f} dB (relative)")
print(f"Noise floor:        {10*np.log10(noise_floor):8.3f} dB (relative)")
print(f"SNR (periodicity):  {snr_db:8.2f} dB {pec_quality_db(snr_db)} signal")

=== Right Ascension - PEC Analysis (N=29) ===
Worm period:          38.679 min
Amplitude:           132.687 arcmin
Peak power:           33.151 dB (relative)
Noise floor:          14.785 dB (relative)
SNR (periodicity):     18.37 dB moderate signal


In [60]:
periods_min = 1 / freqs / 60
log_power = np.log10(power)
max_y = np.max(log_power)
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=periods_min,
    y=log_power,
    mode='lines',
    name='log10(Power)'
))

fig.add_vline(x=worm_period_sec/60, line_dash="dash", line_color="red")

fig.update_layout(
    title="RA Periodogram (Log Power)",
    xaxis_title="Period (minutes)",
    yaxis_title="log10(Power)", 
    height=600, width=1200, template="plotly_dark",
)
max_y
fig.update_yaxes(range=[ -4, max_y])

fig.show()

# Declination Period

In [61]:
from scipy.signal import periodogram
import numpy as np

# Dec: the periodic PEC signal
dec = df['total_accum_2'].values

# Remove linear drift (same idea as DEC fit)
dec_detrended = dec - np.polyval(np.polyfit(t, dec, 1), t)

# Sampling frequency
fs = 1 / np.median(np.diff(t))

# Periodogram
freqs, power = periodogram(dec_detrended, fs=fs)

# Ignore zero frequency and periods > 100 min 
max_period_sec = 120 * 60
min_freq = 1 / max_period_sec
valid = (freqs >= min_freq)
freqs = freqs[valid]
power = power[valid]

# Peak detection
peak_idx = np.argmax(power)
peak_freq = freqs[peak_idx]
peak_power = power[peak_idx]

worm_period_sec = 1 / peak_freq

# Noise floor estimate (median is robust)
mask = np.ones_like(power, dtype=bool)
window = 3  # exclude ±3 bins around peak
mask[max(0, peak_idx-window):peak_idx+window+1] = False
noise_floor = np.median(power[mask])

# Signal-to-noise ratio
snr = peak_power / noise_floor if noise_floor > 0 else np.inf
snr_db = 10 * np.log10(peak_power / noise_floor)

# Estimate amplitude (rough, from detrended signal)
amp = (np.max(ra_detrended) - np.min(ra_detrended)) / 2

# Output
n = len(dec)

print(f"=== Declination - PEC Analysis (N={n}) ===")
print(f"Worm period:        {worm_period_sec/60:8.3f} min")
print(f"Amplitude:          {amp*60:8.3f} arcmin")
print(f"Peak power:         {10*np.log10(peak_power):8.3f} dB (relative)")
print(f"Noise floor:        {10*np.log10(noise_floor):8.3f} dB (relative)")
print(f"SNR (periodicity):  {snr_db:8.2f} dB {pec_quality_db(snr_db)} signal")


=== Declination - PEC Analysis (N=29) ===
Worm period:          38.679 min
Amplitude:           132.687 arcmin
Peak power:           27.365 dB (relative)
Noise floor:           5.945 dB (relative)
SNR (periodicity):     21.42 dB good signal


In [62]:
periods_min = 1 / freqs / 60
log_power = np.log10(power)
max_y = np.max(log_power)
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=periods_min,
    y=log_power,
    mode='lines',
    name='log10(Power)'
))

fig.add_vline(x=worm_period_sec/60, line_dash="dash", line_color="red")

fig.update_layout(
    title="Dec Periodogram (Log Power)",
    xaxis_title="Period (minutes)",
    yaxis_title="log10(Power)", 
    height=600, width=1200, template="plotly_dark",
)
max_y
fig.update_yaxes(range=[ -4, max_y])

fig.show()

# Notes